# CHSH Bell Inequality Violation

Prepares the Bell state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ via the
Qumulator cloud API (H + CNOT circuit, `mode="statevector"`), then measures in four
rotated bases to compute the CHSH correlator $S$.

| Bound | Value |
|-------|-------|
| Classical (local hidden variable) | $S \leq 2$ |
| Tsirelson (quantum maximum) | $S \leq 2\sqrt{2} \approx 2.828$ |

A violation $S > 2$ certifies genuine quantum entanglement — no classical model can reproduce it.

**Protocol:**
1. Submit H + CNOT circuit to Qumulator API — receive exact statevector
2. Apply measurement basis rotations $R_y(\theta)$ locally to the returned statevector
3. Compute $S = E(a,b) + E(a,b') + E(a',b) - E(a',b')$ at optimal CHSH angles
4. Verify $S > 2$ (Bell violation) and $S \leq 2\sqrt{2}$ (Tsirelson bound)

In [ ]:
# ── Set your API key ───────────────────────────────────────────────────────────
# Get a free key: curl -s -X POST https://api.qumulator.com/keys \
#   -H 'Content-Type: application/json' -d '{"name": "my-key"}' | python -m json.tool

import os
API_KEY = os.environ.get('QUMULATOR_API_KEY', 'YOUR_KEY_HERE')
API_URL = 'https://api.qumulator.com'

In [ ]:
%pip install qumulator-sdk --quiet
from qumulator import QumulatorClient
import math
import numpy as np

client = QumulatorClient(api_url=API_URL, api_key=API_KEY)
print('SDK ready.')

## Step 1 — Prepare Bell state via Qumulator API

We submit H + CNOT to the cloud engine with `mode="statevector"` and `return_statevector=True`.
The API returns the exact complex amplitudes $[c_{00}, c_{01}, c_{10}, c_{11}]$.

In [ ]:
bell = client.circuit.run(
    n_qubits=2,
    gates=[
        {"gate": "h",  "qubits": [0]},
        {"gate": "cx", "qubits": [0, 1]},
    ],
    shots=1,
    mode="statevector",
    return_statevector=True,
)

sv = bell.statevector   # numpy array [c00, c01, c10, c11]
print(f'Statevector from API : {np.round(sv, 6)}')
print(f'Probabilities        : {np.round(np.abs(sv)**2, 6)}')
print(f'f_Q_density          : {bell.f_Q_density}')
print(f'entanglement_depth   : {bell.entanglement_depth}')
print(f'phase_label          : {bell.phase_label if hasattr(bell, "phase_label") else "—"}')
assert sv is not None, 'API did not return statevector'
print('\n✓ Bell state received.')

## Step 2 — CHSH correlators from the returned statevector

Apply measurement basis rotations $R_y(\theta_a) \otimes R_y(\theta_b)$ to the API-returned
statevector and compute $E(a,b) = P(00) + P(11) - P(01) - P(10)$.

Optimal CHSH angles: $a=0$, $a'=\pi/2$, $b=\pi/4$, $b'=-\pi/4$.

In [ ]:
I2 = np.eye(2, dtype=complex)

def ry(theta: float) -> np.ndarray:
    c, s = math.cos(theta / 2), math.sin(theta / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)

def apply_1q(state: np.ndarray, gate: np.ndarray, qubit: int) -> np.ndarray:
    full = np.kron(gate, I2) if qubit == 0 else np.kron(I2, gate)
    return full @ state

def correlator(base_sv: np.ndarray, theta_a: float, theta_b: float) -> float:
    """E(a,b) = P(00)+P(11) - P(01)-P(10) after basis rotations Ry(a)⊗Ry(b)."""
    s = apply_1q(base_sv.copy(), ry(theta_a), 0)
    s = apply_1q(s,              ry(theta_b), 1)
    p = np.abs(s) ** 2
    return float(p[0] + p[3] - p[1] - p[2])

# Optimal CHSH angles
a,  a2 = 0.0,          math.pi / 2
b,  b2 = math.pi / 4, -math.pi / 4

E_ab   = correlator(sv, a,  b)
E_ab2  = correlator(sv, a,  b2)
E_a2b  = correlator(sv, a2, b)
E_a2b2 = correlator(sv, a2, b2)

S = E_ab + E_ab2 + E_a2b - E_a2b2

print('CHSH correlators:')
print(f'  E(0,    π/4) = {E_ab:+.6f}  (theory: {math.cos(a  - b ):+.6f})')
print(f'  E(0,   -π/4) = {E_ab2:+.6f}  (theory: {math.cos(a  - b2):+.6f})')
print(f'  E(π/2,  π/4) = {E_a2b:+.6f}  (theory: {math.cos(a2 - b ):+.6f})')
print(f'  E(π/2, -π/4) = {E_a2b2:+.6f}  (theory: {math.cos(a2 - b2):+.6f})')

In [ ]:
tsirelson = 2.0 * math.sqrt(2)
classical = 2.0

print('\n' + '='*55)
print(' BENCHMARK: CHSH Bell Inequality Test')
print('='*55)
print(f'  S (measured)   = {S:.6f}')
print(f'  Classical bound = {classical:.6f}')
print(f'  Tsirelson bound = {tsirelson:.6f}')
print(f'  S > classical  : {"✓ YES — Bell inequality violated" if S > classical else "✗ NO"}')
print(f'  S ≤ Tsirelson  : {"✓ YES" if S <= tsirelson + 1e-9 else "✗ NO"}')

pass_ = (S > classical) and (S <= tsirelson + 0.01)
print(f'\n  Result: {"✓ PASS" if pass_ else "✗ FAIL"}')
assert pass_, f'CHSH test failed: S={S:.6f}'